# 🖼️ Deep Learning Project — CIFAR-10 Image Classifier
## Transfer Learning with ResNet18 (PyTorch)

**Goal:** Build a small image classifier on a CIFAR-10 subset using a pretrained
ResNet18 backbone (transfer learning), with data augmentation.

**Classes:** airplane, cat, dog (3-class subset of CIFAR-10)

**Pipeline:**
1. Load CIFAR-10 image subset (balanced, 300/class train, 100/class test)
2. Data augmentation (flip, rotation, color jitter, random crop)
3. Load pretrained ResNet18, replace classifier head, freeze backbone
4. Train classifier head for several epochs
5. Plot training/validation curves
6. Evaluate: accuracy, precision, recall, F1, confusion matrix
7. Save model checkpoint + provide inference script


## 1. Import Libraries

In [ ]:
import os, time, json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, precision_recall_fscore_support
)
import warnings
warnings.filterwarnings('ignore')

torch.manual_seed(42)
np.random.seed(42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

## 2. Dataset

We use a **3-class subset of CIFAR-10** (`airplane`, `cat`, `dog`) — 32x32 RGB images.

The dataset is organized in `ImageFolder` format:
```
data/CIFAR-10-images-master/
├── train/<class>/*.jpg   (5000 per class originally)
└── test/<class>/*.jpg    (1000 per class originally)
```
We sample a balanced subset: **300 train / 100 test images per class**.


In [ ]:
DATA_ROOT = "data/CIFAR-10-images-master"   # adjust path if needed

SELECTED_CLASSES = ['airplane', 'cat', 'dog']
SAMPLES_PER_CLASS_TRAIN = 300
SAMPLES_PER_CLASS_TEST  = 100
BATCH_SIZE = 32
NUM_EPOCHS = 12
LR = 1e-3
IMG_SIZE = 64   # upscale 32x32 -> 64x64 for ResNet18

print(f"Classes: {SELECTED_CLASSES}")
print(f"Train: {SAMPLES_PER_CLASS_TRAIN} x {len(SELECTED_CLASSES)} = {SAMPLES_PER_CLASS_TRAIN*len(SELECTED_CLASSES)}")
print(f"Test : {SAMPLES_PER_CLASS_TEST} x {len(SELECTED_CLASSES)} = {SAMPLES_PER_CLASS_TEST*len(SELECTED_CLASSES)}")

## 3. Data Augmentation & Transforms

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# Training: augmentation to improve generalisation
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.RandomCrop(IMG_SIZE, padding=4),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# Test: deterministic preprocessing only
test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

print("✅ Transforms defined")

## 4. Build Balanced Dataset Subset

In [ ]:
full_train = datasets.ImageFolder(root=os.path.join(DATA_ROOT, "train"))
full_test  = datasets.ImageFolder(root=os.path.join(DATA_ROOT, "test"))

# class_to_idx maps class name -> ImageFolder index (alphabetical order)
selected_orig_idx = {full_train.class_to_idx[c]: i for i, c in enumerate(SELECTED_CLASSES)}


class TransformedSubset(torch.utils.data.Dataset):
    """Wraps an ImageFolder, restricts to selected indices, remaps labels, applies transform."""
    def __init__(self, base_dataset, indices, transform, label_map):
        self.base = base_dataset
        self.indices = indices
        self.transform = transform
        self.label_map = label_map

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        img, orig_label = self.base[self.indices[i]]
        img = self.transform(img)
        return img, self.label_map[orig_label]


def build_balanced_indices(dataset, per_class, selected_map):
    targets = np.array(dataset.targets)
    indices = []
    for orig_idx in selected_map.keys():
        cls_indices = np.where(targets == orig_idx)[0][:per_class]
        indices.extend(cls_indices.tolist())
    np.random.shuffle(indices)
    return indices


train_indices = build_balanced_indices(full_train, SAMPLES_PER_CLASS_TRAIN, selected_orig_idx)
test_indices  = build_balanced_indices(full_test,  SAMPLES_PER_CLASS_TEST,  selected_orig_idx)

train_dataset = TransformedSubset(full_train, train_indices, train_transform, selected_orig_idx)
test_dataset  = TransformedSubset(full_test,  test_indices,  test_transform,  selected_orig_idx)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"Train set: {len(train_dataset)} images")
print(f"Test set : {len(test_dataset)} images")

### 4.1 Sample Augmented Images

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
sample_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
images, labels = next(iter(sample_loader))
for i, ax in enumerate(axes.flat):
    img = images[i].numpy().transpose(1, 2, 0)
    img = img * np.array(IMAGENET_STD) + np.array(IMAGENET_MEAN)
    img = np.clip(img, 0, 1)
    ax.imshow(img)
    ax.set_title(SELECTED_CLASSES[labels[i]], fontsize=11)
    ax.axis('off')
fig.suptitle('Sample Augmented Training Images', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Transfer Learning Model — ResNet18

We load a **ResNet18** pretrained on ImageNet, freeze its convolutional backbone,
and replace the final fully-connected layer with a new classifier head for our
3-class problem.

> **Note:** If pretrained weights cannot be downloaded (e.g. restricted network),
> the code falls back to random initialization and trains the **entire** network
> from scratch instead of just the head. On a normal machine with internet access,
> `weights=models.ResNet18_Weights.IMAGENET1K_V1` downloads successfully and only
> the head needs to be trained — this is the recommended path.


In [ ]:
print("Loading pretrained ResNet18 (ImageNet weights)...")
PRETRAINED_LOADED = True
try:
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    print("Loaded ImageNet-pretrained weights.")
except Exception as e:
    print(f"Could not download pretrained weights ({e}).")
    print("Falling back to random initialization; training full network from scratch.")
    model = models.resnet18(weights=None)
    PRETRAINED_LOADED = False

# Replace final FC layer for 3-class output
num_features = model.fc.in_features
model.fc = nn.Sequential(
    nn.Linear(num_features, 128),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(128, len(SELECTED_CLASSES))
)

# Freeze backbone ONLY if pretrained weights loaded (true transfer learning)
if PRETRAINED_LOADED:
    for name, param in model.named_parameters():
        if not name.startswith('fc.'):
            param.requires_grad = False

model = model.to(DEVICE)

params_to_train = model.fc.parameters() if PRETRAINED_LOADED else model.parameters()
optimizer = optim.Adam(params_to_train, lr=LR)
criterion = nn.CrossEntropyLoss()
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=4, gamma=0.5)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

## 6. Training Loop

In [ ]:
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
start_time = time.time()

for epoch in range(NUM_EPOCHS):
    # ── Train ──
    model.train()
    running_loss, correct, total_samples = 0.0, 0, 0
    for images, labels in train_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total_samples += labels.size(0)

    train_loss = running_loss / total_samples
    train_acc  = correct / total_samples

    # ── Validate ──
    model.eval()
    val_loss_sum, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss_sum += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)

    val_loss = val_loss_sum / val_total
    val_acc  = val_correct / val_total
    scheduler.step()

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    print(f"Epoch {epoch+1}/{NUM_EPOCHS}  |  "
          f"Train Loss: {train_loss:.4f}  Train Acc: {train_acc:.4f}  |  "
          f"Val Loss: {val_loss:.4f}  Val Acc: {val_acc:.4f}")

elapsed = time.time() - start_time
print(f"\nTraining completed in {elapsed:.1f}s")

## 7. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
epochs_range = range(1, NUM_EPOCHS + 1)

axes[0].plot(epochs_range, history['train_loss'], 'o-', color='#4A90D9', label='Train Loss', lw=2)
axes[0].plot(epochs_range, history['val_loss'], 's-', color='#E67E22', label='Val Loss', lw=2)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('Training & Validation Loss', fontweight='bold')
axes[0].legend()

axes[1].plot(epochs_range, history['train_acc'], 'o-', color='#4A90D9', label='Train Acc', lw=2)
axes[1].plot(epochs_range, history['val_acc'], 's-', color='#E67E22', label='Val Acc', lw=2)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
axes[1].set_title('Training & Validation Accuracy', fontweight='bold')
axes[1].set_ylim(0, 1.05)
axes[1].legend()

plt.tight_layout(); plt.show()

## 8. Final Evaluation on Test Set

In [ ]:
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(DEVICE)
        outputs = model(images)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)

acc  = accuracy_score(all_labels, all_preds)
prec = precision_score(all_labels, all_preds, average='macro')
rec  = recall_score(all_labels, all_preds, average='macro')
f1   = f1_score(all_labels, all_preds, average='macro')

print(f"Accuracy : {acc:.4f}")
print(f"Precision: {prec:.4f} (macro)")
print(f"Recall   : {rec:.4f} (macro)")
print(f"F1 Score : {f1:.4f} (macro)")
print()
print(classification_report(all_labels, all_preds, target_names=SELECTED_CLASSES))

### 8.1 Confusion Matrix

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
cm = confusion_matrix(all_labels, all_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=SELECTED_CLASSES, yticklabels=SELECTED_CLASSES)
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
ax.set_title('Confusion Matrix — Test Set', fontweight='bold')
plt.tight_layout(); plt.show()

### 8.2 Per-Class Metrics

In [ ]:
p, r, f, _ = precision_recall_fscore_support(all_labels, all_preds)

fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(SELECTED_CLASSES))
width = 0.25
ax.bar(x - width, p, width, label='Precision', color='#4A90D9')
ax.bar(x, r, width, label='Recall', color='#E67E22')
ax.bar(x + width, f, width, label='F1', color='#2ECC71')
ax.set_xticks(x); ax.set_xticklabels(SELECTED_CLASSES)
ax.set_ylim(0, 1.1)
ax.set_ylabel('Score')
ax.set_title('Per-Class Metrics', fontweight='bold')
ax.legend()
plt.tight_layout(); plt.show()

## 9. Save Model Checkpoint

In [ ]:
torch.save({
    'model_state_dict': model.state_dict(),
    'classes': SELECTED_CLASSES,
    'img_size': IMG_SIZE,
    'mean': IMAGENET_MEAN,
    'std': IMAGENET_STD,
    'architecture': 'resnet18',
    'pretrained_imagenet_weights': PRETRAINED_LOADED,
}, 'cifar_classifier.pth')

print("✅ Model saved: cifar_classifier.pth")

## 10. Inference on New Images

Use the saved checkpoint to classify a new image. See `predict.py` for a
standalone command-line version: `python predict.py path/to/image.jpg`


In [ ]:
from PIL import Image

def predict_image(image_path, model, transform, classes, device=DEVICE):
    img = Image.open(image_path).convert("RGB")
    tensor = transform(img).unsqueeze(0).to(device)
    model.eval()
    with torch.no_grad():
        outputs = model(tensor)
        probs = torch.softmax(outputs, dim=1)[0]
        pred_idx = torch.argmax(probs).item()
    return classes[pred_idx], probs[pred_idx].item(), {c: round(probs[i].item(),4) for i,c in enumerate(classes)}

# Example: predict on the first test image of each class
for cls in SELECTED_CLASSES:
    sample_path = os.path.join(DATA_ROOT, "test", cls, sorted(os.listdir(os.path.join(DATA_ROOT, "test", cls)))[0])
    pred_class, conf, all_probs = predict_image(sample_path, model, test_transform, SELECTED_CLASSES)
    print(f"True: {cls:10s} → Predicted: {pred_class:10s} (confidence: {conf:.2%})  | {all_probs}")

## 11. Conclusions

- **Architecture:** ResNet18 with a custom 2-layer classifier head (128 hidden units + dropout)
- **Transfer learning:** Backbone frozen, only the head trained (when pretrained weights are available)
- **Data augmentation:** Horizontal flip, rotation, color jitter, random crop — improves generalisation on a small dataset
- **Results:** Test accuracy and macro F1 reported above; confusion matrix shows per-class confusions (notably cat↔dog, which are visually similar at 32x32 resolution)
- **Deliverables:** Trained model saved as `cifar_classifier.pth`, with `predict.py` providing a ready-to-use inference CLI

**Possible improvements:**
- Unfreeze and fine-tune deeper ResNet layers with a low learning rate
- Increase training set size and number of epochs
- Try a larger backbone (ResNet34/50) or EfficientNet
- Add learning-rate warmup and early stopping based on validation loss
